# Kestrel Desk: from chatbot to agent, one harness component at a time

Kestrel Desk is the internal assistant of Kestrel Software (a fictional B2B SaaS company with
sales, engineering and finance teams). We build it in five stages. Every stage is **one
`Agent(...)` call**, and each stage adds the keyword arguments for one more harness component:

| Stage | Name | New keywords |
|---|---|---|
| 0 | Chatbot | `model` |
| 1 | Context | `instructions` |
| 2 | Actions | `tools`, `middleware`, `skills`, `subagents` |
| 3 | Boundaries | `permissions`, `approve`, `audit` |
| 4 | Persistence | `checkpointer`, `workspace`, `compaction` |

**The rhythm.** Run a stage's `def build_stage_N` cell, then its `serve(...)` cell, then
**refresh the chat window** at http://127.0.0.1:8777 and type the prompts
(`chatwin/PROMPTS.md` is the cheat sheet). The `demo(...)` cell runs the same prompt inline.
One story runs through every stage: the Acme Robotics renewal.

In [1]:
from kestrel_chat import serve, demo, list_sessions, get_llm  # notebook helpers (puts the repo on sys.path)
import uuid
from harness.loop import Agent
from harness.config import COMPANY_DIR, SKILLS_DIR, WORKSPACE_DIR, settings
from harness.tools import COMPANY_TOOLS
from harness.middleware import ROLE_TOOLS, CallLimit
from harness.subagents import PROPOSAL_REVIEWER
from harness.persistence import SessionStore
from harness.compaction import Compactor
from harness.talk import describe_stage

store = None  # stage 4: None means SessionStore() on .harness_state/kestrel.sqlite
print("Model:", get_llm().model)

Model: claude-sonnet-5-5


## Stage 0 · Chatbot: Just a model and a prompt

This is where most teams start: a system prompt, a message list and one model call. The model is capable, but it has never heard of Kestrel, our customers or how we write proposals. Let's see what it does with nothing around it.

**What we add:** `model`. A system prompt, a message list, one model call.

In [2]:
def build_stage_0(role="sales", session_id=None, approvals=None):
    llm = get_llm()  # the model from .env (or the offline fake)
    common = dict(role=role, stage=0, session_id=session_id or uuid.uuid4().hex[:12],
                  approvals=approvals)
    # A chatbot: a model and a prompt
    return Agent(model=llm, **common)

In [3]:
serve(build_stage_0, stage=0)

Chat window now serves stage 0 · Chatbot
http://127.0.0.1:8777
-> refresh the chat window


In [4]:
demo(build_stage_0,
     'Who is Acme Robotics and when is their renewal due?')

▶ sales · stage 0 · session 4d311631ac79
  you: Who is Acme Robotics and when is their renewal due?


  assistant:


I don't have any information about an "Acme Robotics" or its renewal date. I can't see your contracts, CRM, or account records in this conversation, and I don't know of a well-known company by that name. "Acme" is also often used as a placeholder name, so it may be an internal or fictional entity.

Here are some ways to find the answer:

- **Contract or CRM system:** Search for the account in Salesforce, HubSpot, or whatever you use. The renewal date is usually on the account or opportunity record.
- **Contract documents:** Check the signed agreement for the term length, start date, and auto-renewal or notice clauses.
- **Billing or finance system:** Look at the subscription or invoice schedule.
- **Account owner or colleagues:** Ask the sales rep, account manager, or procurement contact.

If you paste the contract text, an account summary, or an email thread, I can pull out who they are and work out the renewal date and any notice deadlines. I can also draft a renewal reminder or outreach email if that would help.

  (1 model calls · 0 tool calls · 38 tokens in / 402 out)


## Stage 1 · Context: Tell it who we are

The model knows nothing about Kestrel, so it invents our prices and our rules. AGENTS.md is the always-on company file: who we are, our plans and prices, and the business rules every answer must follow. It rides in the system prompt of every call; nothing else about the call changes.

**What we add:** `instructions`. The company file, always on: facts, prices and rules.

In [5]:
def build_stage_1(role="sales", session_id=None, approvals=None):
    llm = get_llm()  # the model from .env (or the offline fake)
    common = dict(role=role, stage=1, session_id=session_id or uuid.uuid4().hex[:12],
                  approvals=approvals)
    # Context: the company file, always on
    return Agent(model=llm, instructions=COMPANY_DIR / "AGENTS.md", **common)

In [6]:
serve(build_stage_1, stage=1)

Chat window now serves stage 1 · Context
http://127.0.0.1:8777
-> refresh the chat window


In [7]:
demo(build_stage_1,
     'Draft a renewal proposal for Acme Robotics: 40 seats on our Enterprise plan, with pricing.')

▶ sales · stage 1 · session f4ab7dd7a939
  you: Draft a renewal proposal for Acme Robotics: 40 seats on our Enterprise plan, with pricing.


  assistant:


# Renewal Proposal: Acme Robotics

**Subject:** Your Kestrel Desk renewal: Enterprise plan, 40 seats

Hello [Contact name],

Thank you for being a Kestrel Desk customer. Here is our proposal for your upcoming renewal.

## Proposed plan

**Enterprise plan, 40 seats, billed annually**

| Item | Detail |
|---|---|
| Plan | Enterprise |
| Seats | 40 |
| Price per seat | $2,400 / year |
| **Annual total** | **$96,000** |
| Term | 12 months, starting [renewal date] |

## What's included

- Desk Core platform, plus Connect (integrations and API)
- SAML SSO (and standard OIDC SSO)
- Audit log export
- US/EU data residency
- 99.9% uptime SLA
- Named Customer Success Manager

## Next steps

If this looks right, reply to this email and we'll prepare the renewal paperwork. If you'd like to change the seat count, let me know and I'll update the numbers.

Best regards,
[Your name]
Kestrel Software

---

## Notes for you (not part of the email)

- **Pricing** is list price: $2,400 × 40 seats = $96,000. 40 seats clears the 25-seat Enterprise minimum. No discount is included. Anything above 10% needs written finance approval, and multi-year terms need finance sign-off first.
- **Not verified against the customer record:**
  - Acme's current plan, seat count, and renewal date
  - Contact name and `contact_email` (the proposal can only go to the contact on record)
  - Any previously agreed pricing, discounts, or add-ons such as Insights or Services
  - Open issues or incidents that might affect the renewal conversation
- **Sending:** this is a draft only, nothing has been sent. Once the details are confirmed, sending requires human approval and goes only to the contact on record.

  (1 model calls · 0 tool calls · 1,223 tokens in / 896 out)


## Stage 2 · Actions: Let it look things up itself

AGENTS.md can't hold every customer record or every playbook, so we still had to tell it who Acme is. Now the model has to fetch facts and know-how itself: tools for our records, skills it loads only when a task needs them, a reviewer for its draft. A loop that keeps calling tools also needs a brake.

**What we add:** `tools`, `middleware`, `skills`, `subagents`. Tools, the agent loop, middleware, skills on demand and a subagent.

In [8]:
def build_stage_2(role="sales", session_id=None, approvals=None):
    llm = get_llm()  # the model from .env (or the offline fake)
    common = dict(role=role, stage=2, session_id=session_id or uuid.uuid4().hex[:12],
                  approvals=approvals)
    # Actions: tools, a loop with middleware, skills on demand, a subagent
    return Agent(model=llm, instructions=COMPANY_DIR / "AGENTS.md",
                 tools=COMPANY_TOOLS, middleware=[CallLimit()],
                 skills=SKILLS_DIR, subagents=[PROPOSAL_REVIEWER], **common)

In [9]:
serve(build_stage_2, stage=2)

Chat window now serves stage 2 · Actions
http://127.0.0.1:8777
-> refresh the chat window


In [10]:
demo(build_stage_2,
     "Prepare Acme Robotics' renewal proposal: check their record and open issues, then have the proposal-reviewer check it before you show it to me.")

▶ sales · stage 2 · session 501723c36ab7
  you: Prepare Acme Robotics' renewal proposal: check their record and open issues, then have the proposal-reviewer check it before you show it to me.


  load_skill · {"name": "proposal-writing"}


    ✓ load_skill


  lookup_customer · {"name": "Acme Robotics"}


    ✓ lookup_customer


  lookup_issue · {"query": "Acme Robotics"}


... (4 outputs left out)

    ✓ run_subagent


  assistant:


I've drafted the proposal and had the reviewer check it. I have **not** queued it. Tell me if you want it queued once you've approved it.

## Record check
- **Customer (`cust_acme`):** Growth plan, 25 seats, $54,000 ARR. Renewal is 2027-03-31. The contact is Dana Whitfield, procurement@acme-robotics.example.
- **Notes (`cust_acme`):** Acme is evaluating Enterprise for 40 seats. Their security review needs SAML SSO and exportable audit logs. They asked about a multi-year discount, which is not approved.
- **Open issue:** `ISS-1043` (S2, open). Audit log export omits the actor IP column that their security review requires.
- **Resolved issue:** `ISS-1044` (S3). The fleet dashboard slowness was fixed in release 26.9.

## Reviewer feedback and what I changed
- **Pricing and features:** The reviewer said the $2,400 price, the Enterprise features, the 99.9% SLA and the 25-seat minimum weren't in the customer record. They come from our plan table, so I kept them.
- **Audit log export:** Valid point. I qualified it, because the export doesn't yet meet Acme's requirement (`ISS-1043`).
- **Claims I removed:** "Engineering is investigating" and "security review pending on your side." Neither is in the record.

---

# Proposal: Acme Robotics — Enterprise

## Summary
We propose moving Acme Robotics from Growth to Enterprise with 40 seats ahead of the 2027-03-31 renewal. Enterprise includes SAML SSO and audit log export, which your security review calls for.

## Their goals
- Upgrade from Growth to Enterprise for 40 seats (`cust_acme`).
- Meet security review requirements: SAML SSO and exportable audit logs (`cust_acme`).
- Explore a multi-year discount (`cust_acme`).

## Proposed plan
- Plan: Enterprise, 40 seats (the minimum is 25)
- List price: $2,400 per seat per year
- Annual total: 40 × $2,400 = **$96,000**
- Current: Growth, 25 × $2,160 = $54,000 (`cust_acme`)

## What is included
- Everything in Growth, including Connect and standard SSO (OIDC)
- SAML SSO
- Audit log export (see open items)
- US/EU data residency
- 99.9% SLA
- Named CSM

## Open items
- **Multi-year pricing and discounts:** These need finance approval. None is offered in this proposal.
- **`ISS-1043` (S2, open):** Exported audit logs currently omit the actor IP column your security review requires. We can't commit to a fix date.
- **Security review:** Your team's review of SAML SSO and audit log export should take `ISS-1043` into account.

## Next steps
- Dana Whitfield confirms the 40-seat count and the security review timeline.
- Kestrel sales takes the multi-year question to finance.
- Kestrel sales shares `ISS-1043` status updates as they become available.

Prepared by Kestrel Desk for review.

---

**Before sending**
- The 40-seat count comes from the notes, not a confirmed order.
- The `ISS-1043` item is likely to matter most to Acme's security review. Check with engineering before the email goes out.

  (3 model calls · 4 tool calls · 11,844 tokens in / 2,417 out)


## Stage 3 · Boundaries: Draw the lines in code

Now the agent can act, and it treats every user the same. An engineer can pull revenue, anyone can email a customer, and a prompt that says please don't is not a control. Permissions, approvals and an audit trail have to live in the harness, where the model can't talk its way past them.

**What we add:** `permissions`, `approve`, `audit`. Role permissions, human approval, audit log.

In [11]:
def build_stage_3(role="sales", session_id=None, approvals=None):
    llm = get_llm()  # the model from .env (or the offline fake)
    common = dict(role=role, stage=3, session_id=session_id or uuid.uuid4().hex[:12],
                  approvals=approvals)
    # Boundaries: roles, human approval, audit
    return Agent(model=llm, instructions=COMPANY_DIR / "AGENTS.md",
                 tools=COMPANY_TOOLS, middleware=[CallLimit()],
                 skills=SKILLS_DIR, subagents=[PROPOSAL_REVIEWER],
                 permissions=ROLE_TOOLS, approve={"send_proposal"}, audit=True, **common)

In [12]:
serve(build_stage_3, stage=3)

Chat window now serves stage 3 · Boundaries
http://127.0.0.1:8777
-> refresh the chat window


In [13]:
demo(build_stage_3,
     'What was our revenue in FY2026?', role="engineering")

▶ engineering · stage 3 · session 3fa1f41d0e5a
  you: What was our revenue in FY2026?


  get_revenue · {"period": "FY2026"}


    ✗ get_revenue denied: Role 'engineering' is not allowed to use get_revenue. Allowed: list_customers, lookup_customer, lookup_issue.


  assistant:


I can't give you FY2026 revenue. The harness blocked the revenue lookup because the engineering role isn't allowed to use `get_revenue`. I haven't estimated a figure, because I'd only be guessing.

The finance team can pull this. If you ask them, they can also produce a revenue brief with record ids.

I can still help with customer records and issues, such as account background, incident status or triage.

  (2 model calls · 1 tool calls · 5,172 tokens in / 235 out)


## Stage 4 · Persistence: Remember, save, and keep going

Everything so far lives in memory: restart the server and the conversation is gone, and nothing the agent writes survives. Long sessions also keep growing until the prompt is too big and too slow. We need checkpoints to resume, one safe folder to write in, and a way to shrink old turns.

**What we add:** `checkpointer`, `workspace`, `compaction`. Checkpoints, a writable workspace, context compaction.

In [14]:
def build_stage_4(role="sales", session_id=None, approvals=None):
    llm = get_llm()  # the model from .env (or the offline fake)
    common = dict(role=role, stage=4, session_id=session_id or uuid.uuid4().hex[:12],
                  approvals=approvals)
    # Persistence: checkpoints, a workspace, compaction
    return Agent(model=llm, instructions=COMPANY_DIR / "AGENTS.md",
                 tools=COMPANY_TOOLS, middleware=[CallLimit()],
                 skills=SKILLS_DIR, subagents=[PROPOSAL_REVIEWER],
                 permissions=ROLE_TOOLS, approve={"send_proposal"}, audit=True,
                 checkpointer=store or SessionStore(), workspace=WORKSPACE_DIR,
                 compaction=Compactor(at_tokens=settings.compact_at_tokens), **common)

In [15]:
serve(build_stage_4, stage=4)

Chat window now serves stage 4 · Persistence
http://127.0.0.1:8777
-> refresh the chat window


In [16]:
demo(build_stage_4,
     'Draft a short renewal proposal for Acme Robotics (40 seats, Enterprise, SSO and audit logs), save it as workspace/acme-renewal.md, and add a line to company/AGENTS.md saying the Acme draft is done.')

▶ sales · stage 4 · session ddd6497548e3
  you: Draft a short renewal proposal for Acme Robotics (40 seats, Enterprise, SSO and audit logs), save it as workspace/acme-renewal.md, and add a line to company/AGENTS.md saying the Acme draft is done.


  load_skill · {"name": "proposal-writing"}


    ✓ load_skill


  lookup_customer · {"name": "Acme Robotics"}


    ✓ lookup_customer


  lookup_issue · {"query": "cust_acme"}


... (10 outputs left out)

    ✗ read_workspace_file failed: Tool error: Path 'company/AGENTS.md' is outside workspace/. Only workspace/ is writable.


  assistant:


I saved the draft at `workspace/acme-renewal.md`, but I couldn't add the line to `company/AGENTS.md`.

**The AGENTS.md line:** The file system only allows access under `workspace/`, so `company/AGENTS.md` is out of reach. I didn't write the note anywhere else in its place. If you want a note inside the workspace, I can save one as `workspace/acme-status.md`. Otherwise someone with access to `company/` will need to add "Acme draft is done" by hand.

**What the draft says** (`cust_acme`):
- **Change:** Acme moves from Growth (25 seats, $54,000 ARR) to Enterprise (40 seats) at the 2027-03-31 renewal.
- **Price:** 40 × $2,400 = **$96,000** a year at list price.
- **Included:** SAML SSO and audit log export, which Acme's security review requires. The draft also lists the other Enterprise features from our plan table.
- **Recipient:** Dana Whitfield, procurement@acme-robotics.example, the contact on record.

**Open items:**
- **Multi-year discount:** Acme asked about one, and it isn't approved. The draft offers no discount and says finance has to approve one first.
- **Open issues:** I couldn't check Acme's issues because the sales role can't use `lookup_issue`. Engineering should confirm there is no open S1 or S2 before the proposal goes out. The draft says this too.
- **Reviewer flags:** The proposal-reviewer flagged the price, the Enterprise feature list and the 25-seat minimum as unsupported. It could only compare them with the customer record, which has no pricing. All three come from our plan table, so I'm treating them as verified. It also noted that the draft doesn't state a contract term or start date.

I haven't queued anything to the customer. I can queue it with `send_proposal` once you've reviewed it.

  (5 model calls · 6 tool calls · 22,007 tokens in / 2,210 out)


### Pick up yesterday's work

Every message above was checkpointed to SQLite as it happened. `list_sessions()` reads the
saved sessions back (newest first); passing a `session_id` to the builder resumes one, so the
agent answers from its saved history even after a kernel or server restart. In the chat window
the same sessions are the **Past sessions** column; **Compact now** shrinks a long one.

In [17]:
sessions = list_sessions()

ddd6497548e3  sales        12 msgs  2026-11-02 18:35  Draft a short renewal proposal for Acme Robotics (
08d575b427bb  sales        10 msgs  2026-11-02 18:33  Draft a short renewal proposal for Acme Robotics (
003fb22169b4  sales         7 msgs  2026-11-02 18:17  Draft a short renewal proposal for Acme Robotics (


In [18]:
demo(build_stage_4, "Where did we leave off with Acme, and what did we save?",
     session_id=sessions[0]["session_id"] if sessions else None)

↺ resumed session ddd6497548e3 from SQLite (12 messages)


▶ sales · stage 4 · session ddd6497548e3
  you: Where did we leave off with Acme, and what did we save?


  list_workspace_files · {}


    ✓ list_workspace_files


  ⇣ compacted: 15 → 7 messages


  read_workspace_file · {"path": "acme-renewal.md"}


    ✓ read_workspace_file


  assistant:


## Where we left off

The Acme renewal draft is saved and still needs review. Nothing has been queued to the customer. The one file in the workspace is `workspace/acme-renewal.md`.

**What the draft says** (`cust_acme`):
- **Change:** Acme moves from Growth (25 seats) to Enterprise (40 seats) at the 2027-03-31 renewal.
- **Price:** 40 × $2,400 = **$96,000** a year at list price. Current Growth ARR is $54,000.
- **Included:** Desk Core, Connect, SAML SSO, audit log export, US/EU data residency, 99.9% SLA and a named CSM. This matches the Enterprise row of our plan table.
- **Recipient:** Dana Whitfield, procurement@acme-robotics.example, the contact on record.
- **Open items listed in the draft:**
  - The multi-year discount is not approved.
  - Open issues are unchecked.
  - Acme's security review is still pending.

**Still open:**
- **`company/AGENTS.md` note:** Not added. I can only write under `workspace/`, so someone with access needs to add it. I can save a status note in the workspace instead if you'd like.
- **Issue check:** `lookup_issue` is blocked for the sales role. Engineering needs to confirm there is no open S1 or S2 for Acme.
- **Finance:** They need to sign off on the multi-year question.
- **Closing line:** The proposal-writing skill's closing-line instruction was cut off when I loaded it. I used "Prepared by Kestrel Desk for review.", which may not match the required wording.
- **Contract term and start date:** The draft doesn't state either, and the reviewer flagged that.

When you're happy with the draft, I can queue it with `send_proposal` for human approval.

  (3 model calls · 2 tool calls · 16,825 tokens in / 1,265 out)


## Closing: what we built

Five stages, one `Agent(...)` call each. The model never changed: everything that made Kestrel
Desk useful and safe was harness. Context told it who we are, actions let it fetch facts and
know-how itself, boundaries enforced roles and approvals in code, and persistence let it
remember, save and keep going. The chat window keeps running while this kernel is alive.

In [19]:
for n in range(5):
    d = describe_stage(n)
    print(f"Stage {n}  {d['name']:<12} + {', '.join(d['new_keywords'])}")

Stage 0  Chatbot      + model
Stage 1  Context      + instructions
Stage 2  Actions      + tools, middleware, skills, subagents
Stage 3  Boundaries   + permissions, approve, audit
Stage 4  Persistence  + checkpointer, workspace, compaction
